In [1]:
# --- Cell 1: Install TRL (offline auto-detect without internet) ---
import subprocess, sys
from pathlib import Path
try:
    import trl
    print(f"TRL already installed: {trl.__version__}")
except ImportError:
    # Auto-find the uploaded TRL wheel directory in Kaggle inputs
    trl_wheel_dir = "/kaggle/input/trl-offline"  # fallback
    input_dir = Path("/kaggle/input")
    if input_dir.exists():
        for p in input_dir.rglob("trl*none-any.whl"):
            trl_wheel_dir = str(p.parent)
            break

    print(f"Installing TRL offline from: {trl_wheel_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--find-links", trl_wheel_dir, "trl", "--quiet"],
        check=True,
    )
    import trl
    print(f"Installed TRL: {trl.__version__}")

Installing TRL offline from: /kaggle/input/datasets/manishmodak/trl-offline
Installed TRL: 1.1.0


In [2]:
# --- Install bitsandbytes from offline wheels (Kaggle: internet off on accelerator) ---
import subprocess
import sys
from pathlib import Path


def _find_bitsandbytes_findlinks() -> str | None:
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for whl in root.rglob("bitsandbytes*.whl"):
        return str(whl.parent)
    return None


try:
    import bitsandbytes  # noqa: F401

    print("bitsandbytes already importable.")
except ImportError:
    links = _find_bitsandbytes_findlinks()
    if not links:
        print(
            "WARNING: No bitsandbytes wheel under /kaggle/input. "
            "Upload repo folder `bnb_offline/` as a Kaggle Dataset and attach it. "
            "Next cell will fall back to optim=adamw_torch_fused if import still fails."
        )
    else:
        print(f"Installing bitsandbytes (offline) from: {links}")
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--no-index",
                "--find-links",
                links,
                "bitsandbytes",
                "--no-deps",
                "--quiet",
            ],
            check=True,
        )
        print("bitsandbytes pip install finished.")


Installing bitsandbytes (offline) from: /kaggle/input/datasets/manishmodak/bnb-offline
bitsandbytes pip install finished.


In [3]:
# --- Cell 2: Config + Imports ---
import site
import sys
import os
from pathlib import Path

if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# CUTLASS path setup (from NVIDIA utility script)
candidate_cutlass_paths = [
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"),
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/"),
]
for p in candidate_cutlass_paths:
    if p.exists():
        site.addsitedir(str(p))
        print(f"Added cutlass path: {p}")
        break

import pandas as pd
import torch
import kagglehub
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    import mamba_ssm  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "mamba_ssm import failed. Run TRL + bitsandbytes cells, restart kernel, then re-run from Config."
    ) from exc

# ═══════════════════════════════════════════════════════════
# TRAINING CONFIG — Run A-prime (packing + multi-epoch + best checkpoint + longer eval gen)
# ═══════════════════════════════════════════════════════════
OUTPUT_DIR = "/kaggle/working"
FINAL_ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"
MODEL_ID = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
LORA_RANK = 32

# Training knobs (set MAX_TRAIN_SAMPLES for smoke tests)
MAX_TRAIN_SAMPLES = None        # e.g. 256 for a quick run
NUM_EPOCHS = 3                  # reduce to 2 if session time tight; packing lowers steps vs unpadded
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
SAVE_STEPS = 250                # used when USE_EVAL=False (step saves only)
EVAL_STEPS = 250                # trainer eval + best-checkpoint cadence when USE_EVAL=True
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024           # headroom for packed batches; plain rows still short
EVAL_MAX_NEW_TOKENS = 2048      # post-train eval generate (host allows up to ~7680)
LR_SCHEDULER_TYPE = "cosine"    # was "linear" → cosine gives smoother decay
WARMUP_RATIO = 0.05             # was 0 → prevents early loss spikes (10→7 in Run 2)
try:
    import bitsandbytes as _bnb
    _bnb.optim.Adam8bit  # force CUDA init — fails fast if binary missing
    OPTIM = "adamw_8bit"
    print("Using optim=adamw_8bit (bitsandbytes", getattr(_bnb, "__version__", "?"), ")")
except Exception:
    OPTIM = "adamw_torch_fused"
    print("bitsandbytes unavailable or CUDA mismatch — using optim=adamw_torch_fused")

# ═══════════════════════════════════════════════════════════
# EVAL CONFIG
# ═══════════════════════════════════════════════════════════
USE_EVAL = True   # False for final submission → train on all 9,500 rows
SPLIT_DATASET_SLUG = "your-username/nemotron-split-data"  # ← UPDATE after uploading

# Suffix — must match nvidia-nemotron-metric.ipynb exactly
BOXED_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())


Added cutlass path: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Using optim=adamw_8bit (bitsandbytes 0.49.2 )
torch: 2.12.0.dev20260324+cu128 cuda: True


In [4]:
# --- Cell 3: Data loading (eval-split aware) ---
from pathlib import Path

def resolve_train_csv() -> Path:
    """Find the competition train.csv on Kaggle or locally."""
    candidates = [
        Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),
        Path("/kaggle/input/competitions/nvidia-nemotron-3-reasoning-challenge/train.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    found = (
        list(Path("/kaggle/input").rglob("train.csv"))
        if Path("/kaggle/input").exists()
        else []
    )
    if found:
        return found[0]
    local = Path("train.csv")
    if local.exists():
        return local.resolve()
    raise FileNotFoundError(
        "train.csv not found. Add the competition dataset as a Kaggle input."
    )

if USE_EVAL:
    # Auto-find the uploaded split dataset in Kaggle inputs
    split_root = None
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("train_9000.csv"):
            split_root = p.parent
            break
    
    if not split_root:
        # Fallback to local datasets folder
        local_split_root = Path("datasets")
        if local_split_root.exists():
            split_root = local_split_root
        else:
            raise FileNotFoundError(
                "Split data not found. Upload datasets/ folder as a Kaggle dataset or ensure local files exist."
            )

    train_df = pd.read_csv(split_root / "train_9000.csv")
    eval_df = pd.read_csv(split_root / "eval_500.csv")
    print(f"\u2705 Eval mode: Train={len(train_df)}, Eval={len(eval_df)} (from {split_root})")
else:
    # Final submission: use ALL competition data
    train_df = pd.read_csv(resolve_train_csv())
    eval_df = None
    print(f"\U0001f3c1 Final submission mode: Training on ALL {len(train_df)} rows (no eval)")

# Apply sample limit if set
if MAX_TRAIN_SAMPLES is not None:
    train_df = train_df.head(int(MAX_TRAIN_SAMPLES)).copy()
    print(f"\u26a0\ufe0f Truncated to {len(train_df)} samples (MAX_TRAIN_SAMPLES={MAX_TRAIN_SAMPLES})")

print(f"Training rows: {len(train_df)}")

✅ Eval mode: Train=9000, Eval=500 (from /kaggle/input/datasets/manishmodak/nemotron-split-data)
Training rows: 9000


In [5]:
# --- Cell 4: Tokenizer + data formatting (TRL 1.x Prompt-Completion format) ---
import os
from pathlib import Path

def resolve_model_path() -> str:
    """Find the Nemotron model on Kaggle inputs or download via kagglehub."""
    direct_candidates = [
        Path("/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default"),
        Path("/kaggle/input/nemotron-3-nano/transformers/default/1"),
    ]
    for path in direct_candidates:
        if path.exists():
            print(f"Found model in Kaggle inputs: {path}")
            return str(path)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for config_path in input_root.rglob("config.json"):
            candidate = config_path.parent
            candidate_text = str(candidate).lower()
            if "nemotron" not in candidate_text:
                continue
            if not (candidate / "tokenizer_config.json").exists():
                continue
            print(f"Found model in Kaggle inputs: {candidate}")
            return str(candidate)

    print("Downloading model via kagglehub (requires internet)...")
    return kagglehub.model_download(MODEL_ID)

MODEL_PATH = resolve_model_path()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

def format_prompt(prompt: str) -> str:
    user_content = str(prompt).strip() + BOXED_SUFFIX
    messages = [{"role": "user", "content": user_content}]
    kwargs = dict(tokenize=False, add_generation_prompt=True)
    try:
        return tokenizer.apply_chat_template(messages, **kwargs, enable_thinking=True)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)

def format_completion(answer: str) -> str:
    # Note: TRL 1.1 automatically appends eos_token if needed, but we provide the raw generation
    return f"\\boxed{{{str(answer).strip()}}}"

# Build training dataset as prompt-completion dictionaries
prompts = []
completions = []
for row in train_df.itertuples(index=False):
    prompts.append(format_prompt(row.prompt))
    completions.append(format_completion(row.answer))

train_ds = Dataset.from_dict({"prompt": prompts, "completion": completions})

eval_ds = None
if eval_df is not None:
    _ep = [format_prompt(r.prompt) for r in eval_df.itertuples(index=False)]
    _ec = [format_completion(r.answer) for r in eval_df.itertuples(index=False)]
    eval_ds = Dataset.from_dict({"prompt": _ep, "completion": _ec})
    print(f"Trainer eval dataset: {len(eval_ds)} examples (held-out; eval_loss + best checkpoint)")

print(f"Training dataset: {len(train_ds)} examples")
print(f"\n--- Sample prompt (first 500 chars) ---")
print(train_ds[0]["prompt"][:500])
print(f"\n--- Sample completion ---")
print(train_ds[0]["completion"])

Found model in Kaggle inputs: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Trainer eval dataset: 500 examples (held-out; eval_loss + best checkpoint)
Training dataset: 9000 examples

--- Sample prompt (first 500 chars) ---
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
\{@^> = '&^
[&:[{ = ^'`{
^^|\\ = |&&
<>@`' = '{>
Now, determine the result for: \\@>>
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`<|im_end|>
<|im_start|>assistant
<think>


--- Sample completion ---
\boxed{'>&}


In [6]:
# --- Cell 7: Load 30B model + attach LoRA (rank 32) ---
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [7]:
# --- Cell 8: Training (SFTTrainer with response-only loss) ---
from trl import SFTTrainer, SFTConfig
import shutil
from pathlib import Path
import subprocess

os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

# ─── SFTConfig (Run A-prime: packing + optional best checkpoint on eval_loss) ───
_sft_common = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    optim=OPTIM,
    logging_steps=50,
    ignore_data_skip=True,
    bf16=True,
    report_to="none",
    dataset_text_field=None,
    completion_only_loss=True,
    max_length=MAX_SEQ_LENGTH,
    packing=True,
)

if eval_ds is not None:
    sft_config = SFTConfig(
        **_sft_common,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        per_device_eval_batch_size=1,
        save_strategy="best",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        load_best_model_at_end=True,
        save_total_limit=1,
    )
else:
    sft_config = SFTConfig(
        **_sft_common,
        eval_strategy="no",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=1,
    )

# ─── Create trainer ───
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

# ─── Fix Kaggle read-only Triton binary permissions ───
triton_bin_candidates = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin",
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin",
]
ro_bin_dir = next((path for path in triton_bin_candidates if os.path.exists(path)), None)
rw_bin_dir = "/kaggle/working/triton_bin"
if ro_bin_dir:
    os.makedirs(rw_bin_dir, exist_ok=True)
    for f in os.listdir(ro_bin_dir):
        src = os.path.join(ro_bin_dir, f)
        dst = os.path.join(rw_bin_dir, f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        os.chmod(dst, 0o777)

    orig_popen = subprocess.Popen

    def patched_popen(*args, **kwargs):
        if args and isinstance(args[0], list) and isinstance(args[0][0], str):
            if args[0][0].startswith(ro_bin_dir):
                args[0][0] = args[0][0].replace(ro_bin_dir, rw_bin_dir)
        return orig_popen(*args, **kwargs)

    subprocess.Popen = patched_popen

# ─── Print config summary ───
print("=" * 60)
print("TRAINING CONFIG (Run A-prime)")
print("=" * 60)
print(f"  Epochs:         {NUM_EPOCHS}")
print(f"  LR:             {LEARNING_RATE}")
print(f"  LR schedule:    {LR_SCHEDULER_TYPE} (was linear)")
print(f"  Warmup ratio:   {WARMUP_RATIO} (was 0)")
print(f"  Optimizer:      {OPTIM}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Post-train eval: max_new_tokens={EVAL_MAX_NEW_TOKENS}")
print(f"  Batch size:     {PER_DEVICE_TRAIN_BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} grad accum")
print(f"  Response-only:  {getattr(sft_config, 'completion_only_loss', True)}")
print(f"  Packing:        {getattr(sft_config, 'packing', False)}")
print(f"  Train samples:  {len(train_ds)}")
print(f"  Save:           {'best@eval_loss' if eval_ds is not None else 'steps=' + str(SAVE_STEPS)}, total_limit=1")
print("=" * 60)

# Resume from prior session (attach previous output / dataset with checkpoint-*)
resume_path = None
_inp = Path("/kaggle/input")
if _inp.exists():
    for _cp in sorted(_inp.rglob("checkpoint-*"), reverse=True):
        if (_cp / "trainer_state.json").exists():
            _dest = Path("/kaggle/working") / _cp.name
            if not _dest.exists():
                shutil.copytree(str(_cp), str(_dest))
            resume_path = str(_dest)
            print("Resuming from:", resume_path)
            break
if resume_path is None:
    print("No checkpoint in /kaggle/input - training from scratch")

trainer.train(resume_from_checkpoint=resume_path)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/9000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/9000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

Packing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


TRAINING CONFIG (Run A-prime)
  Epochs:         3
  LR:             0.0002
  LR schedule:    cosine (was linear)
  Warmup ratio:   0.05 (was 0)
  Optimizer:      adamw_8bit
  Max seq length: 1024
  Post-train eval: max_new_tokens=2048
  Batch size:     2 x 4 grad accum
  Response-only:  True
  Packing:        True
  Train samples:  9000
  Save:           best@eval_loss, total_limit=1
No checkpoint in /kaggle/input - training from scratch


Step,Training Loss,Validation Loss
250,1.230702,0.345097
500,0.907801,0.347346


TrainOutput(global_step=561, training_loss=1.6293244489374008, metrics={'train_runtime': 10220.8404, 'train_samples_per_second': 0.439, 'train_steps_per_second': 0.055, 'total_flos': 8.546136279120219e+17, 'train_loss': 1.6293244489374008})

In [8]:
import shutil
# --- Cell 9: Save trained adapter weights ---
from pathlib import Path

# Free disk: remove training checkpoint(s) before saving final adapter
import glob as _gl
for _ckpt in _gl.glob(f"{OUTPUT_DIR}/checkpoint-*"):
    shutil.rmtree(_ckpt, ignore_errors=True)
    print(f"Removed {_ckpt}")

final_adapter_dir = Path(FINAL_ADAPTER_DIR)
final_adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print("Saved adapter + tokenizer under", final_adapter_dir)


Removed /kaggle/working/checkpoint-250
Saved adapter + tokenizer under /kaggle/working/final_adapter


In [9]:
# --- Cell 10: Zip adapter files for Kaggle submission ---
import subprocess
from pathlib import Path

out = Path(OUTPUT_DIR)
adapter_dir = Path(FINAL_ADAPTER_DIR)
submission_zip = out / "submission.zip"
cfg = adapter_dir / "adapter_config.json"
weights = adapter_dir / "adapter_model.safetensors"
if not weights.exists():
    weights = adapter_dir / "adapter_model.bin"
if not cfg.exists() or not weights.exists():
    raise FileNotFoundError(f"Missing adapter files. Expected {cfg} and a weights file.")

if submission_zip.exists():
    submission_zip.unlink()

subprocess.run([
    "zip",
    "-j",
    str(submission_zip),
    str(cfg),
    str(weights),
], check=True)
print("Wrote", submission_zip)

  adding: adapter_config.json (deflated 55%)
  adding: adapter_model.safetensors (deflated 9%)
Wrote /kaggle/working/submission.zip


In [10]:

# --- Cell 11: Evaluation on 500 held-out rows ---
import json
import re
import math
from pathlib import Path

if eval_df is not None:
    print("=" * 60)
    print(f"RUNNING EVALUATION on {len(eval_df)} held-out rows")
    print("=" * 60)

    # ─── Family classification (keyword-based) ───
    def classify_family(prompt_text: str) -> str:
        p = str(prompt_text).lower()
        if "bit manipulation" in p:
            return "bit_manipulation"
        if "gravitational constant" in p:
            return "gravity"
        if "encryption rules" in p or "encryption" in p:
            return "encryption"
        if "numeral system" in p:
            return "numeral"
        if "unit conversion" in p or "converted" in p:
            return "unit_conversion"
        if "transformation rules" in p or "equation" in p or "operator" in p:
            return "equations"
        return "unknown"

    # ─── Answer extraction (matches nvidia-nemotron-metric.ipynb) ───
    def extract_final_answer(text):
        if text is None:
            return "NOT_FOUND"
        matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
        if matches:
            non_empty = [m.strip() for m in matches if m.strip()]
            if non_empty:
                return non_empty[-1]
            return matches[-1].strip()
        patterns = [
            r'The final answer is:\s*([^\n]+)',
            r'Final answer is:\s*([^\n]+)',
            r'Final answer\s*[:：]\s*([^\n]+)',
            r'final answer\s*[:：]\s*([^\n]+)',
        ]
        for pattern in patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            if matches:
                return matches[-1].strip()
        matches = re.findall(r'-?\d+(?:\.\d+)?', text)
        if matches:
            return matches[-1]
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return lines[-1] if lines else "NOT_FOUND"

    # ─── Verification (matches nvidia-nemotron-metric.ipynb) ───
    def verify(stored_answer, predicted):
        stored_answer = str(stored_answer).strip()
        predicted = str(predicted).strip()
        if re.fullmatch(r'[01]+', stored_answer):
            return predicted.lower() == stored_answer.lower()
        try:
            stored_num = float(stored_answer)
            predicted_num = float(predicted)
            return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
        except Exception:
            return predicted.lower() == stored_answer.lower()

    # ─── Run inference ───
    model.eval()
    correct = 0
    total = 0
    family_stats = {}
    failures = []

    for i, row in enumerate(eval_df.itertuples(index=False)):
        # Format prompt (same as metric: user message + generation prompt)
        user_content = str(row.prompt).strip() + BOXED_SUFFIX
        messages = [{"role": "user", "content": user_content}]
        _kw = dict(tokenize=False, add_generation_prompt=True)
        try:
            prompt_text = tokenizer.apply_chat_template(messages, **_kw, enable_thinking=True)
        except TypeError:
            prompt_text = tokenizer.apply_chat_template(messages, **_kw)

        inputs = tokenizer(
            prompt_text, return_tensors="pt", truncation=True, max_length=2048
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=EVAL_MAX_NEW_TOKENS,
                temperature=1.0,
                top_p=1.0,
                do_sample=False,
            )

        # Decode only the generated (new) tokens
        generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        predicted = extract_final_answer(generated_text)
        ground_truth = str(row.answer).strip()
        is_correct = verify(ground_truth, predicted)

        # Track per-family stats
        family = classify_family(row.prompt)
        if family not in family_stats:
            family_stats[family] = {"correct": 0, "total": 0}
        family_stats[family]["total"] += 1
        if is_correct:
            correct += 1
            family_stats[family]["correct"] += 1
        elif len(failures) < 10:
            failures.append({
                "id": row.id if hasattr(row, "id") else i,
                "family": family,
                "expected": ground_truth,
                "predicted": predicted,
                "generated": generated_text[:200],
            })

        total += 1
        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{len(eval_df)}] accuracy so far: {correct}/{total} ({100*correct/total:.1f}%)")

    # ─── Print results ───
    print("\n" + "=" * 60)
    print(f"EVAL RESULTS ({total} held-out rows)")
    print("=" * 60)
    print(f"Overall accuracy: {correct}/{total} ({100*correct/total:.1f}%)")
    print("\nPer-family breakdown:")
    for fam, stats in sorted(family_stats.items()):
        pct = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
        print(f"  {fam:20s}: {stats['correct']}/{stats['total']} ({pct:.1f}%)")
    if failures:
        print(f"\nSample failures (first {min(len(failures), 5)}):")
        for f in failures[:5]:
            print(f"  [{f['family']}] expected={f['expected']!r} got={f['predicted']!r}")
    print("=" * 60)

    # ─── Save eval_results.json ───
    results = {
        "overall": {
            "correct": correct,
            "total": total,
            "accuracy": correct / total if total > 0 else 0,
        },
        "per_family": family_stats,
        "failures": failures,
    }
    eval_json_path = Path(OUTPUT_DIR) / "eval_results.json"
    with open(eval_json_path, "w") as f_out:
        json.dump(results, f_out, indent=2)
    print(f"\nSaved eval results to {eval_json_path}")

else:
    print("Eval disabled (USE_EVAL=False). Skipping eval.")


RUNNING EVALUATION on 500 held-out rows


  [50/500] accuracy so far: 28/50 (56.0%)
  [100/500] accuracy so far: 57/100 (57.0%)
  [150/500] accuracy so far: 88/150 (58.7%)
  [200/500] accuracy so far: 110/200 (55.0%)
  [250/500] accuracy so far: 136/250 (54.4%)
  [300/500] accuracy so far: 168/300 (56.0%)
  [350/500] accuracy so far: 191/350 (54.6%)
  [400/500] accuracy so far: 217/400 (54.2%)
  [450/500] accuracy so far: 249/450 (55.3%)
  [500/500] accuracy so far: 278/500 (55.6%)

EVAL RESULTS (500 held-out rows)
Overall accuracy: 278/500 (55.6%)

Per-family breakdown:
  bit_manipulation    : 36/84 (42.9%)
  encryption          : 29/83 (34.9%)
  equations           : 13/82 (15.9%)
  gravity             : 33/84 (39.3%)
  numeral             : 83/83 (100.0%)
  unit_conversion     : 84/84 (100.0%)

Sample failures (first 5):
  [encryption] expected='bird writes door' got='king writes door'
  [encryption] expected='cat studies the silver message' got='cat studies the clever message'
  [encryption] expected='rabbit studies near t